In [1]:
from llms.genserv.client import GenerationServiceClient
from evalserv_client import EvaluationServiceClient
from collections import Counter
from tasks import get_task
import json, time

eval_client = EvaluationServiceClient(base_url=f"http://localhost:5001")
assistant_gen_client = GenerationServiceClient(base_url=f"http://localhost:5000")

assistant_gen_client.load_model("microsoft/phi-4", max_concurrent_jobs_per_worker=45)

dataset_fn = "data/sharded_instructions_600.json"
with open(dataset_fn, "r") as f:
    data = json.load(f)

data = [d for d in data if d["task"] == "code"]
id2sample = {d["task_id"]: d for d in data}

In [ ]:
from collections import Counter
import tqdm

def generate_responses(conversation, group_size):
    # Schedule all generation jobs in batch
    jobs = [{"conversation": conversation, "n_responses": 1} for _ in range(group_size)]
    batch_result = assistant_gen_client.schedule_job_batch(jobs)
    active_job_ids = batch_result["job_ids"]
    
    responses = []
    eval_job_id2response = {}
    active_eval_job_ids = []

    while active_job_ids or active_eval_job_ids:
        # Check all active generation jobs in batch
        if active_job_ids:
            gen_batch_results = assistant_gen_client.check_job_batch(active_job_ids)
            completed_gen_jobs = []
            evaluations_to_schedule = []
            
            for job_id, job_result in gen_batch_results["results"].items():
                if job_result["status"] == "completed":
                    response = job_result["responses"][0]
                    completed_gen_jobs.append(job_id)
                    this_conversation = conversation + [{"role": "assistant", "content": response["response_text"]}]
                    evaluations_to_schedule.append({"conversation": this_conversation, "task_name": sample["task"], "sample": sample})
                    responses.append(response)
            
            # Schedule evaluation jobs in batch
            if evaluations_to_schedule:
                eval_batch_result = eval_client.schedule_evaluation_batch(evaluations_to_schedule)
                eval_job_ids = eval_batch_result["job_ids"]
                for eval_job_id, response in zip(eval_job_ids, responses[-len(eval_job_ids):]):
                    eval_job_id2response[eval_job_id] = response
                    active_eval_job_ids.append(eval_job_id)
            
            # Remove completed generation jobs
            for job_id in completed_gen_jobs:
                active_job_ids.remove(job_id)
        
        # Check all active evaluation jobs in batch
        if active_eval_job_ids:
            eval_batch_results = eval_client.check_job_batch(active_eval_job_ids)
            completed_eval_jobs = []
            
            for job_id, job_result in eval_batch_results["results"].items():
                if job_result["status"] == "completed" and "evaluation_return" in job_result.get("result", {}):
                    completed_eval_jobs.append(job_id)
                    response = eval_job_id2response[job_id]
                    response["score_og"] = job_result["result"]["evaluation_return"]["score"]
                elif job_result["status"] == "error" or (job_result["status"] == "completed" and "evaluation_return" not in job_result.get("result", {})):
                    completed_eval_jobs.append(job_id)
                    response = eval_job_id2response[job_id]
                    response["score_og"] = 0
            
            # Remove completed evaluation jobs
            for job_id in completed_eval_jobs:
                active_eval_job_ids.remove(job_id)
        
        time.sleep(0.1)
    return responses

task_ids = ["sharded-livecodebench/2756", "sharded-livecodebench/2755", "sharded-livecodebench/2847", "sharded-livecodebench/2786", "sharded-livecodebench/2791", "sharded-livecodebench/2856", "sharded-livecodebench/2857", "sharded-livecodebench/2866", "sharded-livecodebench/2882", "sharded-livecodebench/2883"]
RESPONSES = {}
N_responses = 1000

for task_id in tqdm.tqdm_notebook(task_ids):
    sample = id2sample[task_id]
    task = get_task(sample["task"])

    system_message = task.generate_system_prompt(sample)
    input_prompt = task.populate_fully_specific_prompt(sample)

    conversation = [{"role": "system", "content": system_message}, {"role": "user", "content": input_prompt}]
    RESPONSES[task_id] = generate_responses(conversation, N_responses)


with open("all_responses.json", "w") as f:
    json.dump(RESPONSES, f)

# original unbatched genserv: 13min16seconds

/tmp/ipykernel_1120208/2954003504.py:44: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  for task_id in tqdm.tqdm_notebook(task_ids):


  0%|          | 0/10 [00:00<?, ?it/s]

# Rerun evaluations

In [ ]:
from evalserv_client import EvaluationServiceClient
from collections import Counter
from tasks import get_task
import tqdm
import ujson as json, time

dataset_fn = "data/sharded_instructions_600.json"
with open(dataset_fn, "r") as f:
    data = json.load(f)

data = [d for d in data if d["task"] == "code"]
id2sample = {d["task_id"]: d for d in data}

task = get_task("code")

eval_client = EvaluationServiceClient(base_url=f"http://localhost:5001")

# num_workers, timeout = 400, 3
# num_workers, timeout = 200, 3 # done
# num_workers, timeout = 100, 3 # done
# num_workers, timeout = 50, 3 # done
num_workers, timeout = 400, 6
# num_workers, timeout = 200, 6 # done
# num_workers, timeout = 100, 6 # done
# num_workers, timeout = 50, 6 # done

T_start = time.time()
eval_key = f"score_w{num_workers}_t{timeout}"

with open("all_responses.json", "r") as f:
    RESPONSES = json.load(f)

# Prepare all evaluation jobs for batch scheduling
evaluations_batch = []
eval_job_id2response = {}
response_order = []

for task_id, responses in RESPONSES.items():
    sample = id2sample[task_id]
    system_message = task.generate_system_prompt(sample)
    input_prompt = task.populate_fully_specific_prompt(sample)
    conversation = [{"role": "system", "content": system_message}, {"role": "user", "content": input_prompt}]
    for response in responses:
        full_conversation = conversation + [{"role": "assistant", "content": response["response_text"]}]
        evaluations_batch.append({"conversation": full_conversation, "task_name": "code", "sample": sample})
        response_order.append(response)

# Schedule all evaluation jobs in batch
print(f"Scheduling {len(evaluations_batch)} evaluations in batch...")
batch_result = eval_client.schedule_evaluation_batch(evaluations_batch)
job_ids = batch_result["job_ids"]

# Create mapping from job_id to response
for job_id, response in zip(job_ids, response_order):
    eval_job_id2response[job_id] = response

# Poll for results using batch checking
active_job_ids = set(job_ids)
progress_bar = tqdm.tqdm_notebook(total=len(active_job_ids), desc="Evaluating responses")
completed_count = 0

while active_job_ids:
    # Check all active jobs in batch
    batch_results = eval_client.check_job_batch(list(active_job_ids))
    jobs_to_remove = []
    
    for job_id, job_result in batch_results["results"].items():
        if job_result["status"] == "completed" and "evaluation_return" in job_result.get("result", {}):
            jobs_to_remove.append(job_id)
            response = eval_job_id2response[job_id]
            response[eval_key] = job_result["result"]["evaluation_return"]["score"]
        elif job_result["status"] == "error" or (job_result["status"] == "completed" and "evaluation_return" not in job_result.get("result", {})):
            jobs_to_remove.append(job_id)
            response = eval_job_id2response[job_id]
            response[eval_key] = 0
    
    # Update progress and remove completed jobs
    if jobs_to_remove:
        for job_id in jobs_to_remove:
            active_job_ids.remove(job_id)
        new_completed = len(jobs_to_remove)
        completed_count += new_completed
        progress_bar.update(new_completed)
    
    time.sleep(0.1)

progress_bar.close()

T_end = time.time()
print(f"[Num workers: {num_workers}, Timeout: {timeout}] Time taken: {T_end - T_start} seconds")

# Save updated responses
with open("all_responses.json", "w") as f:
    json.dump(RESPONSES, f)


# [Num workers: 400, Timeout: 3] Time taken: 273.0245752334595 seconds
# [Num workers: 200, Timeout: 3] Time taken: 323.70234775543213 seconds
# [Num workers: 100, Timeout: 3] Time taken: 384.0455656051636 seconds
# [Num workers: 50, Timeout: 3] Time taken: 516.7160129547119 seconds

# [Num workers: 400, Timeout: 6] Time taken: 281.4258587360382 seconds
# [Num workers: 200, Timeout: 6] Time taken: 334.1034679412842 seconds
# [Num workers: 100, Timeout: 6] Time taken: 441.6714069843292 seconds
# [Num workers: 50, Timeout: 6] Time taken: 583.9686925411224 seconds


Scheduling 10000 evaluations in batch...


/tmp/ipykernel_2667436/69049856.py:59: TqdmDeprecationWarning: This function will be removed in tqdm==5.0.0
Please use `tqdm.notebook.tqdm` instead of `tqdm.tqdm_notebook`
  progress_bar = tqdm.tqdm_notebook(total=len(active_job_ids), desc="Evaluating responses")


Evaluating responses:   0%|          | 0/10000 [00:00<?, ?it/s]

[Num workers: 400, Timeout: 6] Time taken: 281.4258587360382 seconds


In [2]:
import ujson as json, pandas as pd, numpy as np

with open("all_responses.json", "r") as f:
    RESPONSES = json.load(f)

print(len(RESPONSES))

task_ids = list(RESPONSES.keys())

eval_keys = sorted([k for k in RESPONSES[task_ids[0]][0].keys() if k.startswith("score_")])

task_id2row = {task_id: {"task_id": task_id, "num_responses": len(RESPONSES[task_id])} for task_id in task_ids}

for task_id in task_ids:
    row = task_id2row[task_id]
    for eval_key in eval_keys:
        avg_score = np.mean([r[eval_key] for r in RESPONSES[task_id]])
        # print(f"{task_id} {eval_key} {avg_score}")
        row[eval_key] = avg_score

dataset = pd.DataFrame(list(task_id2row.values()))

# Apply gradient color styling: each row independently, brighter blue for higher values
dataset.style.background_gradient(cmap='Blues', axis=1, subset=[col for col in dataset.columns if col.startswith('score_')])


10


,task_id,num_responses,score_og,score_w100_t3,score_w100_t6,score_w200_t3,score_w200_t6,score_w400_t3,score_w400_t6,score_w50_t3,score_w50_t6
0,sharded-livecodebench/2756,1000,0.566000,0.566000,0.566000,0.566000,0.566000,0.566000,0.566000,0.566000,0.566000
1,sharded-livecodebench/2755,1000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000
2,sharded-livecodebench/2847,1000,0.859000,0.859000,0.859000,0.859000,0.859000,0.859000,0.859000,0.859000,0.859000
3,sharded-livecodebench/2786,1000,0.269000,0.269000,0.269000,0.269000,0.269000,0.269000,0.269000,0.269000,0.269000
4,sharded-livecodebench/2791,1000,0.484000,0.484000,0.484000,0.484000,0.484000,0.484000,0.484000,0.484000,0.484000
5,sharded-livecodebench/2856,1000,0.641000,0.531000,0.633000,0.531000,0.535000,0.528000,0.531000,0.539000,0.702000
6,sharded-livecodebench/2857,1000,0.244000,0.244000,0.244000,0.244000,0.244000,0.244000,0.244000,0.244000,0.244000
7,sharded-livecodebench/2866,1000,0.707000,0.707000,0.707000,0.707000,0.707000,0.707000,0.707000,0.707000,0.707000
8,sharded-livecodebench/2882,1000,0.393000,0.361000,0.388000,0.338000,0.368000,0.336000,0.339000,0.371000,0.394000
9,sharded-livecodebench/2883,1000,0.746000,0.746000,0.746000,0.746000,0.746000,0.746000,0.746000,0.746000,0.746000
